# Hospital Readmissions Analytics — CMS HRRP FY 2026

**Dr. Natheer Soliman, MD**  
Medical Doctor developing expertise in Healthcare Data Analytics, Clinical Analytics, and Medical AI

## 1. Project Overview

This notebook analyzes hospital-condition performance records from the CMS Hospital Readmissions Reduction Program (HRRP) for FY 2026. It focuses on data quality, reporting completeness, condition-level patterns, state-level summaries, and persistent multi-condition hospital signals.

> Educational portfolio analysis only. An Excess Readmission Ratio (ERR) is a risk-adjusted performance signal—not a complete hospital-quality ranking and not evidence of causation.


## 2. Clinical / Healthcare Question

How do reported excess readmission signals vary across HRRP conditions, states, and hospitals, and which facilities show a consistent signal across at least five reported conditions?


## 3. Dataset

**Official source:** CMS Provider Data Catalog — Hospital Readmissions Reduction Program  
https://data.cms.gov/provider-data/dataset/9n3s-kdb3

**Unit of analysis:** one hospital-condition record.  
**Performance period:** July 1, 2021 through June 30, 2024 for FY 2026 reporting.

### Kaggle setup

Attach the official CMS CSV as a Kaggle input, or upload it to the notebook session. This notebook searches Kaggle input folders and common local paths without embedding or republishing the raw source file.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")

EXPECTED_FILE = "FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv"
OFFICIAL_CMS_CSV = "https://data.cms.gov/provider-data/sites/default/files/resources/a171bc36c488d3e0dc33ec63abb469a6_1770163617/FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv"
candidates = [
    Path(EXPECTED_FILE),
    Path("data") / EXPECTED_FILE,
    Path("/content") / EXPECTED_FILE,
]
kaggle_candidates = list(Path("/kaggle/input").glob(f"**/{EXPECTED_FILE}")) if Path("/kaggle/input").exists() else []
data_path = next((p for p in [*candidates, *kaggle_candidates] if p.exists()), None)

if data_path is None:
    print("Local copy not found; loading the official CMS CSV directly.")
    df_raw = pd.read_csv(OFFICIAL_CMS_CSV)
    print(f"Loaded: {OFFICIAL_CMS_CSV}")
else:
    df_raw = pd.read_csv(data_path)
    print(f"Loaded: {data_path}")
print(f"Shape: {df_raw.shape}")
df_raw.head()


## 4. Data Quality

Before analysis, inspect schema, duplicates, missingness, footnotes, measure coverage, and suppressed values. Suppressed values such as “Too Few to Report” must not be converted to zero.


In [ ]:
required = {
    "Facility Name", "Facility ID", "State", "Measure Name",
    "Number of Discharges", "Footnote", "Excess Readmission Ratio",
    "Predicted Readmission Rate", "Expected Readmission Rate",
    "Number of Readmissions", "Start Date", "End Date",
}
missing_columns = sorted(required - set(df_raw.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

quality_summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean().mul(100).round(1),
})
display(quality_summary)
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("\nMeasure counts:")
display(df_raw["Measure Name"].value_counts(dropna=False).to_frame("records"))
print("\nFootnote counts:")
display(df_raw["Footnote"].value_counts(dropna=False).to_frame("records"))


## 5. Data Cleaning

Cleaning preserves source fields and creates separate analysis-ready variables. Non-numeric readmission entries are coerced to missing—not zero—and dates are parsed explicitly.


In [ ]:
MEASURE_MAP = {
    "READM-30-AMI-HRRP": "Acute Myocardial Infarction (AMI)",
    "READM-30-HF-HRRP": "Heart Failure (HF)",
    "READM-30-COPD-HRRP": "COPD",
    "READM-30-PN-HRRP": "Pneumonia",
    "READM-30-CABG-HRRP": "CABG",
    "READM-30-HIP-KNEE-HRRP": "Hip/Knee Replacement",
}

df = df_raw.copy()
df["Number of Readmissions Numeric"] = pd.to_numeric(
    df["Number of Readmissions"], errors="coerce"
)
df["Condition"] = df["Measure Name"].map(MEASURE_MAP)
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce")
df["End Date"] = pd.to_datetime(df["End Date"], errors="coerce")

unmapped = df.loc[df["Condition"].isna(), "Measure Name"].dropna().unique()
if len(unmapped):
    print("Unmapped measure names:", unmapped)

print("Analysis-ready shape:", df.shape)


## 6. Exploratory Data Analysis


In [ ]:
valid_err = df["Excess Readmission Ratio"].dropna()

kpis = pd.Series({
    "hospital_condition_records": len(df),
    "unique_hospitals": df["Facility ID"].nunique(),
    "states_or_territories": df["State"].nunique(),
    "clinical_conditions": df["Condition"].nunique(),
    "valid_ERR_records": len(valid_err),
    "median_ERR": valid_err.median(),
    "ERR_above_1_n": int((valid_err > 1).sum()),
    "ERR_above_1_pct": (valid_err > 1).mean() * 100,
})
display(kpis.to_frame("value"))

condition_summary = (
    df.groupby("Condition")
      .agg(
          Records=("Facility ID", "size"),
          Valid_ERR=("Excess Readmission Ratio", "count"),
          Mean_ERR=("Excess Readmission Ratio", "mean"),
          Median_ERR=("Excess Readmission Ratio", "median"),
          Mean_Predicted_Rate=("Predicted Readmission Rate", "mean"),
          Mean_Expected_Rate=("Expected Readmission Rate", "mean"),
      )
)
condition_summary["ERR_Above_1_Pct"] = (
    df[df["Excess Readmission Ratio"].notna()]
      .groupby("Condition")["Excess Readmission Ratio"]
      .apply(lambda x: (x > 1).mean() * 100)
)
display(condition_summary.round(3).sort_values("Mean_ERR", ascending=False))


In [ ]:
plot_data = condition_summary.sort_values("Mean_Predicted_Rate")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data.index, plot_data["Mean_Predicted_Rate"], color="#146C94")
ax.set(
    title="Average Predicted 30-Day Readmission Rate by Condition",
    xlabel="Mean predicted readmission rate (%)",
    ylabel="Clinical condition",
)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(valid_err, bins=35, kde=True, color="#0B5A75", ax=ax)
ax.axvline(1.0, color="#C0392B", linestyle="--", label="ERR = 1")
ax.set(title="Distribution of Valid Excess Readmission Ratios", xlabel="Excess Readmission Ratio")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Analysis

### State-level summaries

State means are descriptive aggregates of available hospital-condition records. They are not adjusted comparisons between state populations.


In [ ]:
state_summary = (
    df.groupby("State")
      .agg(
          Hospitals=("Facility ID", "nunique"),
          Valid_ERR=("Excess Readmission Ratio", "count"),
          Mean_ERR=("Excess Readmission Ratio", "mean"),
          Median_ERR=("Excess Readmission Ratio", "median"),
          Mean_Predicted_Rate=("Predicted Readmission Rate", "mean"),
          Mean_Expected_Rate=("Expected Readmission Rate", "mean"),
      )
)
state_summary["ERR_Above_1_Pct"] = (
    df[df["Excess Readmission Ratio"].notna()]
      .groupby("State")["Excess Readmission Ratio"]
      .apply(lambda x: (x > 1).mean() * 100)
)

display(
    state_summary[state_summary["Valid_ERR"] >= 100]
    .sort_values("Mean_ERR", ascending=False)
    .head(10)
    .round(3)
)


### Persistent multi-condition hospital signals

To reduce unstable rankings based on one or two reported measures, this analysis limits the persistent-signal view to hospitals with at least five valid conditions.


In [ ]:
valid = df[df["Excess Readmission Ratio"].notna()].copy()

hospital_summary = (
    valid.groupby(["Facility ID", "Facility Name", "State"])
         .agg(
             Conditions_Reported=("Condition", "nunique"),
             Mean_ERR=("Excess Readmission Ratio", "mean"),
             Median_ERR=("Excess Readmission Ratio", "median"),
             Conditions_ERR_Above_1=("Excess Readmission Ratio", lambda x: int((x > 1).sum())),
             Conditions_ERR_Below_1=("Excess Readmission Ratio", lambda x: int((x < 1).sum())),
         )
         .reset_index()
)

robust = hospital_summary[hospital_summary["Conditions_Reported"] >= 5].copy()
persistent_high = robust[
    robust["Conditions_ERR_Above_1"] == robust["Conditions_Reported"]
].sort_values("Mean_ERR", ascending=False)
persistent_low = robust[
    robust["Conditions_ERR_Below_1"] == robust["Conditions_Reported"]
].sort_values("Mean_ERR")

print("Hospitals with valid ERR:", len(hospital_summary))
print("Hospitals with >=5 valid conditions:", len(robust))
print("Persistent high-ERR hospitals:", len(persistent_high))
print("Persistent low-ERR hospitals:", len(persistent_low))

display(persistent_high.head(10).round(3))
display(persistent_low.head(10).round(3))


## 8. Evaluation

This is descriptive hospital-level analytics, not a patient prediction model. Evaluation therefore focuses on:

- reproducible record counts and source-period checks;
- explicit handling of missing and suppressed values;
- minimum reporting coverage for hospital-level signals;
- comparison of predicted, expected, and excess-ratio summaries;
- transparent interpretation boundaries.

The notebook should not report accuracy, sensitivity, or ROC-AUC for the HRRP layer because no patient-level classifier is being evaluated here.


## 9. Healthcare Interpretation

- **ERR > 1** means observed risk-adjusted performance is higher than expected for that condition under the HRRP methodology.
- It does **not** prove poor care, causation, or the effectiveness of a specific intervention.
- A multi-condition signal may help prioritize review, but hospital case mix, data completeness, measure construction, and local context still matter.
- Operational follow-up should connect analytics to discharge planning, medication reconciliation, access to follow-up, and care-transition workflows.


## 10. Limitations

1. HRRP records are hospital-condition aggregates, not patient encounters.
2. Suppressed and missing values reduce comparable coverage.
3. Averaging ERR across conditions is a descriptive portfolio choice, not an official CMS composite score.
4. Cross-sectional FY 2026 reporting does not establish trends or causality.
5. Results may change when CMS refreshes the source dataset.
6. This analysis does not assess social risk, coding variation, or local intervention capacity.


## 11. Conclusion

This workflow demonstrates how to turn an official hospital-quality dataset into transparent, reproducible healthcare analytics without confusing hospital-level performance signals with patient-level risk. The main portfolio value is not a ranking; it is the disciplined handling of units, missingness, coverage, and interpretation.


## 12. References / Data Source

1. Centers for Medicare & Medicaid Services. *Hospital Readmissions Reduction Program* dataset: https://data.cms.gov/provider-data/dataset/9n3s-kdb3
2. CMS HRRP program overview: https://www.cms.gov/medicare/quality/value-based-programs/hospital-readmissions
3. GitHub project: https://github.com/natheerne-hub/Hospital-Readmissions-Healthcare-Analytics
4. Live MVP: https://hospital-readmissions-healthcare-an.vercel.app
